# Eksplorasi hasil evaluasi: Gemini vs BLIP

Notebook ini membaca hasil `ai-service/eval` lalu menampilkan tabel ringkas akurasi dan latensi per run.

- Sumber data dan cara hitungnya sama dengan `plotting/buat_grafik.py` (modul `plotting/data_evaluasi.py`): berkas
  prediksi `ai-service/eval/data/predictions/*.jsonl` bila ada (tidak masuk git), bila tidak ada memakai tabel di
  `ai-service/eval/data/reports/laporan.md`.
- Bila belum ada hasil evaluasi sama sekali, notebook menampilkan pesan dan tetap selesai tanpa error.
- Jalankan dengan kernel dari venv `ai-service` (`pip install -r requirements-dev.txt`).

In [ ]:
import sys
from pathlib import Path

# Cari akar repo (folder yang berisi plotting/) dari lokasi kerja notebook.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "plotting" / "data_evaluasi.py").is_file())
sys.path.insert(0, str(ROOT / "plotting"))
import data_evaluasi

hasil = data_evaluasi.muat()
if hasil is None:
    print(data_evaluasi.PESAN_KOSONG)
else:
    print(f"Sumber data: {hasil['sumber']} · {len(hasil['runs'])} run")

## Ringkasan per run

In [ ]:
from IPython.display import Markdown, display

def persen(x):
    return "–" if x is None else f"{x * 100:.1f}%"

def detik(ms):
    return "–" if ms is None else f"{ms / 1000:.1f} dtk"

if hasil is None:
    display(Markdown("_Belum ada data. Lihat pesan di atas._"))
else:
    baris = ["| Run | Provider | Berhasil | Dinilai manusia | Akurat | Jenis benar | p50 | p95 | ≤ 5 dtk |",
             "|---|---|---|---|---|---|---|---|---|"]
    for r in hasil["runs"]:
        jenis = f"{persen(r['jenis_acc'])} (n={r['jenis_n']})" if r["jenis_n"] else "– (tidak memberi jenis)"
        baris.append(f"| {r['run']} | {r['provider']} | {r['n_ok']}/{r['n_attempted']} | {r['n_rated']} | {persen(r['acc'])} | "
                     f"{jenis} | {detik(r['p50'])} | {detik(r['p95'])} | {persen(r['fast'])} |")
    display(Markdown("\n".join(baris)))

## Gemini vs BLIP (per provider)

Tiap provider diwakili run dengan jumlah gambar berhasil terbanyak.

In [ ]:
if hasil is not None:
    terbaik = {}
    for r in hasil["runs"]:
        if r["p50"] is not None and r["n_ok"] >= terbaik.get(r["provider"], {}).get("n_ok", -1):
            terbaik[r["provider"]] = r
    baris = ["| Provider | Run | p50 | p95 | ≤ 5 dtk (RNF-04) | Jenis benar | Akurat (manusia) |", "|---|---|---|---|---|---|---|"]
    for prov, r in terbaik.items():
        baris.append(f"| {prov} | {r['run']} | {detik(r['p50'])} | {detik(r['p95'])} | {persen(r['fast'])} | "
                     f"{persen(r['jenis_acc'])} | {persen(r['acc']) if r['n_rated'] else 'belum dinilai'} |")
    display(Markdown("\n".join(baris)))

## Per jenis gambar

Butuh berkas prediksi. Kolom *Akurat* baru terisi setelah penilaian manusia (`python -m eval.annotate`).

In [ ]:
if hasil is None or not hasil["per_jenis"]:
    print("Rincian per jenis gambar belum tersedia (butuh ai-service/eval/data/predictions/*.jsonl).")
else:
    baris = ["| Run | Jenis (acuan) | Gambar | Memberi jenis | Jenis benar | Dinilai | Akurat |", "|---|---|---|---|---|---|---|"]
    for g in hasil["per_jenis"]:
        baris.append(f"| {g['run']} | {g['jenis']} | {g['n']} | {g['memberi_jenis']} | {g['jenis_benar']} | {g['n_dinilai']} | {g['akurat']} |")
    display(Markdown("\n".join(baris)))

## Grafik akhir

Grafik dibuat oleh `plotting/buat_grafik.py` ke `docs/gambar/evaluasi/`. Sel di bawah hanya menampilkannya bila sudah ada.

In [ ]:
from IPython.display import Image

folder = ROOT / "docs" / "gambar" / "evaluasi"
gambar = sorted(folder.glob("*.png")) if folder.is_dir() else []
if not gambar:
    print("Belum ada grafik. Jalankan: python -I plotting/buat_grafik.py")
for p in gambar:
    print(p.relative_to(ROOT).as_posix())
    display(Image(filename=str(p)))